### human in loop 

In [ ]:
# --- HumanInTheLoopMiddleware: approval gates on specific tools (NOTES.md §4, §7) ---
# This is the create_agent (middleware) way to get human-in-the-loop, vs. the
# hand-built interrupt() in 02-agents/1-BasicChatbot/2-humanintheloop.ipynb. interrupt_on
# pauses the agent before the named tool runs so a human can approve/edit/reject it.
from langgraph.types import Checkpointer
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.messages import HumanMessage

def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recepient:str, subject:str, body:str)->str:
    """Mock function to send an email."""
    return f"Email sent to for ID: {recepient} with subject '{subject}'"

model=init_chat_model("ollama:qwen3:8b")
agent=create_agent(
    model=model,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),   # required to pause and resume (NOTES.md §5)
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # gate the risky action (sending) behind human approval...
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False,   # ...but let the safe read run freely
            }
        )
    ]
)

config={"configurable":{"thread_id":"test_approve"}}
# This invoke pauses instead of finishing: the result will contain "__interrupt__"
# (handled in the next cell), because send_email_tool needs approval.
result=agent.invoke(
    {"messages":[HumanMessage(content="send email to chris@nolan.com with subject 'Hello' and body 'Sup?'")]}, config
)

In [ ]:
# If the run paused, "__interrupt__" is present. Resume by passing the human's
# decision via Command(resume=...) — approve / reject / or (here) edit the tool args
# before it runs. This is the same resume mechanism as the hand-built interrupt()
# (NOTES.md §4); the middleware just standardizes the decision payload.
from langgraph.types import Command
print(result)
if "__interrupt__" in result:
    print("--------PAUSED! Reviewing---------")
    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    # {"type":"approve"}   # run send_email_tool as the model proposed
                    # {"type":"reject"}    # block it and tell the model it was rejected
                    {
                        "type":"edit",     # change the args, then run with the edits
                        "edited_action":{
                            "name":"send_email_tool",
                            "args":{
                                "recepient":"chris@jenner.com",
                                "subject":"anetolli",
                                "body":"good steal"
                            }
                        }
                    }
                ]
            }
        ),config
    )
    print(f"Result: {result["messages"][-1].content}")